# Week 10, Part 1: APIs from Python
**Phase 2: Python for Data Analysis**

Until now your data came from generators and files. Real analytics starts when you **pull data yourself**: token prices from CoinGecko, protocol TVL from DefiLlama, transaction history from Etherscan. This week has three notebooks:

| Notebook | Topic | You'll be able to |
|---|---|---|
| **`lesson.ipynb`** (this one) | HTTP & APIs | Call any REST API reliably: auth, errors, retries, rate limits, pagination, caching |
| `lesson_expanded.ipynb` | SQLite & SQL from Python | Design a local database, load it safely, query with SQL and Pandas |
| `lesson_advanced.ipynb` | The pipeline | Wire both into a re-runnable, incremental ETL: the engine of the Phase 2 capstone |

**Offline-first.** Every cell here runs **without internet or API keys**, using `fixtures.py`, a stand-in that copies the *shape* of the real responses (field names, nesting, string-typed numbers, errors hidden in HTTP 200, 429 with `Retry-After`). When you're ready, flip one flag (`LIVE = True`) and the *same code* hits the real APIs. Section 13 explains how, plus a smoke-test script to verify your setup.

> ⚠️ **Honest limits.** The fixture numbers are simulated, and the fixtures were built from the providers' public docs, not recorded live traffic. APIs change. When something breaks live, trust the official docs (links in `RESOURCES.md`) over this notebook.

**Prerequisites:** Weeks 5 (functions), 7 (files/errors/JSON), 8 (Pandas).

## 0. Setup

In [ ]:
import json
import os
import sys
import time
import itertools
import tempfile
import pathlib

import numpy as np
import pandas as pd
import requests

sys.path.insert(0, "../../phase-1-python-fundamentals/week-05-functions-modules")
from blockchain_utils import fmt_usd, fmt_large, shorten_address        # your Week 5 toolkit

from api_clients import (ApiError, RateLimitError, RateLimiter, request_json, make_live_session,
                         CoinGeckoClient, DefiLlamaClient, EtherscanClient, tx_dataframe)
from fixtures import FixtureSession, TX_ADDRESS

LIVE = False        # False = offline fixtures (default). True = real APIs (see section 13).

session = make_live_session() if LIVE else FixtureSession()
def limiter(rate):                  # offline: no real waiting. Live: obey the real rate.
    return RateLimiter(rate) if LIVE else RateLimiter(10_000)

cg = CoinGeckoClient(session, limiter=limiter(0.5))
dl = DefiLlamaClient(session, limiter=limiter(2))
es = EtherscanClient(session, api_key=os.environ.get("ETHERSCAN_API_KEY", "demo"), limiter=limiter(3))
print("mode:", "LIVE" if LIVE else "OFFLINE fixtures")

## 1. What an API call actually is

An **API** (here, a REST web API) is a program that answers questions over HTTP. You send a **request**; it sends back a **response**. Nothing more.

```
YOU (Python)                                        SERVER (CoinGecko, Etherscan...)
   | -- REQUEST ---------------------------------------> |
   |    method : GET                                       |
   |    URL    : https://api.coingecko.com/api/v3/simple/price
   |    query  : ?ids=ethereum&vs_currencies=usd           |   (the "WHERE clause")
   |    headers: x-cg-demo-api-key: ...  (who you are)     |
   |                                                       |
   | <-- RESPONSE --------------------------------------- |
   |    status : 200                                       |   (did it work?)
   |    headers: Content-Type: application/json, ...       |
   |    body   : {"ethereum": {"usd": 3250.0}}             |   (the data)
```

Four things to internalise:

1. **Method**: `GET` reads data (what you'll use 95% of the time). `POST` sends data (submitting a transaction, querying with a body). `GET` is **safe to retry**; some `POST`s are not.
2. **URL = address + query string.** Everything after `?` is parameters: `key=value` pairs joined by `&`.
3. **Status code** is the first thing to check, before you touch the body.
4. **Body** is usually JSON, which Python reads as `dict`/`list`.

### 1.1 The Dune parallel

| Dune / SQL | API |
|---|---|
| A table (`dex.trades`) | An **endpoint** (`/coins/markets`) |
| `WHERE chain = 'ethereum'` | Query parameter `?chain=ethereum` |
| `LIMIT 100 OFFSET 200` | Pagination parameters (`per_page=100&page=3`) |
| Query credits / timeout | **Rate limits** / HTTP 429 |
| Query result rows | JSON body |
| Your Dune account | An **API key** |

The big difference: a SQL engine is **flexible** (any query you can write) while an API is **fixed** (only the endpoints the provider built). That's why you'll often fetch a bit more than you need and filter locally with Pandas.

### 1.2 Seeing a request before you send it

`requests` can build a request **without sending it**. Do this whenever a URL looks wrong: you see exactly what would go over the wire.

In [ ]:
req = requests.Request(
    "GET",
    "https://api.coingecko.com/api/v3/simple/price",
    params={"ids": "ethereum,bitcoin", "vs_currencies": "usd"},
    headers={"x-cg-demo-api-key": "YOUR_KEY_HERE"},
).prepare()

print("method :", req.method)
print("url    :", req.url)                 # params were encoded into the query string for you
print("headers:", dict(req.headers))

Now send a real call through our session (offline: the fixture) and inspect the response object. The three attributes you'll use constantly are `status_code`, `headers` and `.json()`:

In [ ]:
resp = session.get("https://api.coingecko.com/api/v3/simple/price",
                   params={"ids": "ethereum,bitcoin", "vs_currencies": "usd"}, timeout=15)

print("status :", resp.status_code, "| ok:", resp.ok)
print("json   :", resp.json())
print(type(resp.json()))

## 2. JSON is Python dicts and lists

JSON is a text format; `resp.json()` parses it into native Python objects:

| JSON | Python | Example from our APIs |
|---|---|---|
| object `{}` | `dict` | `{"ethereum": {"usd": 3250.0}}` |
| array `[]` | `list` | the list of protocols |
| string | `str` | **Etherscan returns numbers as strings** |
| number | `int` / `float` | CoinGecko prices |
| `true` / `false` | `True` / `False` | |
| `null` | `None` | a missing field |

Navigating nested JSON is just chained indexing. `json.dumps(obj, indent=2)` pretty-prints anything, and it's the fastest way to *understand an unfamiliar response*:

In [ ]:
raw = request_json(session, "https://api.llama.fi/protocol/lido")
print("top-level keys :", list(raw.keys()))
print("tvl is a       :", type(raw["tvl"]).__name__, "with", len(raw["tvl"]), "points")
print(json.dumps(raw["tvl"][:2], indent=2))                      # first two points, pretty
print("nested access  :", raw["chainTvls"]["Ethereum"]["tvl"][-1])

**First move with any new API:** call one endpoint, `print(json.dumps(x, indent=2)[:1500])`, and *read it*. Which keys are there? Which are strings that should be numbers? Which timestamps, and in what unit? Ten minutes here saves hours of debugging later.

## 3. Status codes: what happened, and what to do

| Code | Meaning | Your move |
|---|---|---|
| **200** | OK | Parse the body. (**But Etherscan can say 200 and still fail, see section 7.**) |
| **400** | Bad request (a parameter is wrong) | Fix your request. Retrying won't help |
| **401 / 403** | Missing/invalid key, or not allowed | Check the key/plan. Don't retry |
| **404** | No such endpoint/resource | Check the URL or id |
| **429** | **Too many requests** | Wait (the `Retry-After` header says how long), then retry |
| **500, 502, 503, 504** | Server problem | Wait, then retry with backoff |

The rule that drives everything in this notebook: **429 and 5xx are transient, so retry them; other 4xx are your bug, so fail loudly and fast.**

Our `request_json` helper (in `api_clients.py`) already implements it. Watch it handle two failures:

In [ ]:
# 404: an unknown coin id. Client error -> ApiError immediately, no retry.
try:
    request_json(session, "https://api.coingecko.com/api/v3/coins/not-a-coin/market_chart",
                 params={"vs_currency": "usd", "days": 7})
except ApiError as e:
    print("404 ->", e, "| status:", e.status, "| retryable:", e.retryable)

# 401: a bad API key.
try:
    request_json(session, "https://api.coingecko.com/api/v3/ping", headers={"x-cg-demo-api-key": "bad"})
except ApiError as e:
    print("401 ->", e, "| status:", e.status, "| retryable:", e.retryable)

## 4. Query parameters done right

Never build URLs by gluing strings. Pass a `params` dict and let `requests` encode it: it handles special characters, spaces and ordering for you.

In [ ]:
# The tempting way: easy to get subtly wrong
bad = "https://api.coingecko.com/api/v3/coins/markets?vs_currency=usd&per_page=" + str(5) + "&page=" + str(2)

# The right way
good = requests.Request("GET", "https://api.coingecko.com/api/v3/coins/markets",
                        params={"vs_currency": "usd", "per_page": 5, "page": 2}).prepare().url
print(bad == good, "(same here, but only the second survives a value like 'a b&c')")

tricky = requests.Request("GET", "https://example.com/x", params={"q": "a b&c", "ids": ["x", "y"]}).prepare().url
print(tricky)           # space and & are escaped; a list becomes repeated keys (ids=x&ids=y)

Provider conventions differ. CoinGecko wants **comma-joined** ids (`ids=ethereum,bitcoin`), not a repeated key, so you join the list yourself (`",".join(ids)`). Booleans are usually the **strings** `"true"`/`"false"`. Read each endpoint's docs for the exact form.

## 5. JSON to DataFrame: the three shapes you'll meet

Nearly every blockchain API returns one of three shapes. Recognise the shape and the Pandas code writes itself.

### 5.1 Shape A: list of records (the easy one)
DefiLlama's `/protocols` is a list of dicts, which is **exactly** a table: one dict = one row.

In [ ]:
df = pd.DataFrame(request_json(session, "https://api.llama.fi/protocols"))
df[["name", "category", "tvl", "change_1d"]].sort_values("tvl", ascending=False).head(5)
# SQL: SELECT name, category, tvl, change_1d FROM protocols ORDER BY tvl DESC LIMIT 5

### 5.2 Shape B: a record with a list inside
`/protocol/lido` is *one* object whose `tvl` key holds a list of daily points. Select the inner list, then build the frame. For deeper nesting, `pd.json_normalize` flattens nested dicts into dotted column names.

In [ ]:
raw = request_json(session, "https://api.llama.fi/protocol/lido")
tvl = pd.DataFrame(raw["tvl"])                         # [{'date': ..., 'totalLiquidityUSD': ...}, ...]
print(tvl.head(2))
print()
print(pd.json_normalize({"id": 1, "chain": {"name": "Ethereum", "id": 1}, "tvl": 28e9}))   # nested -> flat columns

### 5.3 Shape C: parallel arrays (CoinGecko's favourite)
`market_chart` returns `prices`, `market_caps` and `total_volumes` as **separate lists of `[timestamp, value]` pairs**. You rebuild the table by lining them up. This is what `CoinGeckoClient.market_chart()` does for you; here it is by hand:

In [ ]:
raw = request_json(session, "https://api.coingecko.com/api/v3/coins/ethereum/market_chart",
                   params={"vs_currency": "usd", "days": 7, "interval": "daily"})
print({k: f"{len(v)} pairs" for k, v in raw.items()})
print("one pair:", raw["prices"][0])

prices = pd.DataFrame(raw["prices"], columns=["ts_ms", "price_usd"])
prices["volume_usd"] = [v for _, v in raw["total_volumes"]]       # same length, same order
prices["date"] = pd.to_datetime(prices["ts_ms"], unit="ms", utc=True)
prices

### 5.4 The timestamp-unit trap

| Provider | Unit | `pd.to_datetime(..., unit=)` |
|---|---|---|
| CoinGecko | **milliseconds** | `"ms"` |
| DefiLlama | seconds | `"s"` |
| Etherscan | seconds (as a **string**) | `"s"` after `astype(int)` |

Use the wrong unit and you get either a wildly wrong year (pandas 3) or an error (pandas 2). **Silent is worse**, so add a sanity guard after every timestamp parse:

In [ ]:
ms = 1_790_812_800_000
print("right:", pd.to_datetime(ms, unit="ms", utc=True))

# Wrong unit: what happens depends on your pandas version!
try:
    print("wrong:", pd.to_datetime(ms, unit="s", utc=True), "  <- pandas 3: silently returns an absurd date")
except Exception as e:
    print("wrong unit ->", type(e).__name__, " <- pandas 2: raises an error")

def check_sane(dates, lo="2009-01-01", hi="2100-01-01"):
    # Cheap guard: blockchain data cannot predate Bitcoin. Call it after every timestamp parse.
    bad = ((dates < pd.Timestamp(lo, tz="UTC")) | (dates > pd.Timestamp(hi, tz="UTC"))).sum()
    if bad:
        raise ValueError(f"{bad} timestamps outside {lo}..{hi}: wrong unit (s vs ms)?")
    return dates

check_sane(pd.Series([pd.to_datetime(ms, unit="ms", utc=True)]))           # passes
try:
    check_sane(pd.Series([pd.to_datetime(ms, unit="s", utc=True)]))
except ValueError as e:
    print("guard caught it:", e)

## 6. Authentication and secrets

An **API key** identifies you so the provider can apply *your* quota. Keys have two enemies: **leaks** and **sloppy logging**.

### 6.1 Where does the key go?

| Provider | How the key is sent | Note |
|---|---|---|
| **CoinGecko** (Demo plan) | Header `x-cg-demo-api-key` (recommended) or query `x_cg_demo_api_key` | The docs recommend the header because query strings can leak into logs and history |
| **Etherscan V2** | Query parameter `apikey` | One key for all chains via `chainid` |
| **DefiLlama** (free) | **No key** | The Pro API embeds the key in the URL path |

> 📖 **Verified against the official docs (Oct 2026):** CoinGecko's [authentication page](https://docs.coingecko.com/demo/reference/authentication) documents the header method; Etherscan's [V2 docs](https://docs.etherscan.io/etherscan-v2/rate-limits) document the free tier at 5 calls/second and 100,000 calls/day.

> ⚠️ **Update to the course notes:** the project notes said CoinGecko needs no key. It now offers a **free Demo plan with a key**; the keyless public API still exists but is heavily throttled. Our client works either way and uses a key if you provide one.

### 6.2 Never hardcode keys

```python
API_KEY = "CG-abc123..."          # ❌ ends up on GitHub; bots scrape GitHub for keys within minutes
```

Read from the **environment** instead:

In [ ]:
os.environ["DEMO_ONLY_KEY"] = "CG-fake-key-123456"            # in real life you set this in your shell, not in code
key = os.environ.get("DEMO_ONLY_KEY")                          # .get() returns None if unset (no crash)
print("key present:", bool(key))

def redact(url, secret_params=("apikey", "x_cg_demo_api_key")):
    '''Hide secrets before logging a URL. Etherscan puts the key IN the URL, so logging URLs leaks it.'''
    from urllib.parse import urlsplit, urlunsplit, parse_qsl, urlencode
    p = urlsplit(url)
    q = [(k, "***" if k in secret_params else v) for k, v in parse_qsl(p.query)]
    return urlunsplit((p.scheme, p.netloc, p.path, urlencode(q, safe="*"), p.fragment))

print(redact("https://api.etherscan.io/v2/api?chainid=1&module=account&apikey=SECRET123"))

### 6.3 The `.env` file workflow

Typing `export` every session is annoying. The standard solution is a **`.env` file** that is **never committed**:

```bash
# .env   (in the repo root; add ".env" to .gitignore FIRST)
ETHERSCAN_API_KEY=your_key_here
COINGECKO_DEMO_API_KEY=CG-your_key_here
```

```python
# pip install python-dotenv
from dotenv import load_dotenv
load_dotenv()                     # reads .env into os.environ
```

```bash
# .gitignore additions
.env
*.db
.cache/
```

**If a key leaks** (you pushed it, even briefly): treat it as compromised. **Revoke/regenerate it in the provider dashboard immediately.** Deleting the commit doesn't help; git history and scrapers have it. (This is also a good moment to check your repo's uncommitted files for things like `api_config.json` before running `git add .`.)

## 7. Errors: not every failure is a status code

### 7.1 Etherscan answers HTTP 200 even when it fails

This catches almost everyone once. Etherscan's HTTP status is `200` for a bad key, a bad address, a rate limit, a missing endpoint. The truth is in the **JSON body**:

```json
{"status": "0", "message": "NOTOK", "result": "Missing/Invalid API Key"}
```

`raise_for_status()` would happily pass this. You must check `status` in the body:

In [ ]:
resp = session.get("https://api.etherscan.io/v2/api",
                   params={"chainid": 1, "module": "account", "action": "balance",
                           "address": TX_ADDRESS, "tag": "latest", "apikey": ""})
print("HTTP status :", resp.status_code, "  <- looks fine!")
print("body        :", resp.json())

# Our client converts body-level failures into exceptions:
try:
    EtherscanClient(session, api_key="", limiter=limiter(3)).balance_eth(TX_ADDRESS)
except ApiError as e:
    print("client raised:", e)

Also note: Etherscan returns **"No transactions found"** with `status "0"` for an address with no history. That's an *empty result*, not an error, and the client treats it that way.

### 7.2 Error shapes differ per provider

| Provider | Failure looks like |
|---|---|
| CoinGecko | Real HTTP 4xx/429 with a JSON `error`/`status` body |
| DefiLlama | Real HTTP 4xx/5xx; some missing resources return a 400 |
| Etherscan | **HTTP 200** + `status:"0"` + reason in `result` |

So each client normalises its provider's quirks into one exception type, `ApiError`, with `.status`, `.retryable` and `.url`. **Your analysis code should never need to know provider quirks.** That's the point of writing clients instead of scattering `requests.get` calls through notebooks.

### 7.3 Always set a timeout

`requests` waits **forever** by default. One hung connection freezes your whole pipeline. Every call gets `timeout=` (our helper defaults to 15 s). A tuple `(connect, read)` such as `(3.05, 15)` lets you be stricter about connecting than reading.

## 8. Retries and exponential backoff

Transient failures (a 503 blip, a 429 when you called too fast) usually succeed a moment later. The wrong response is to crash; the other wrong response is to **hammer** the server immediately, which is what turns a blip into a ban.

**Exponential backoff** doubles the wait each time: 1 s, 2 s, 4 s, 8 s. Early retries are quick (most blips pass fast); later ones back off (a real outage gets breathing room). If the server tells you exactly how long (`Retry-After`), **obey that instead.**

Production systems also add **jitter** (a small random amount) so that 1,000 clients who all failed together don't all retry at the same instant. We omit it to keep results deterministic; add `random.uniform(0, 0.3)` when you deploy.

Let's watch it work. The fixture can be told to fail the first calls:

In [ ]:
waits = []
flaky = FixtureSession(flaky=[(429, {"Retry-After": "3"}), (503, {}), (503, {})])

data = request_json(flaky, "https://api.llama.fi/protocols", sleep=waits.append)    # sleep=waits.append: record, don't wait
print("succeeded after", len(flaky.calls), "calls")
print("waited (s):", waits)
print("   429 + Retry-After:3 -> obeyed (3.0)")
print("   503 attempt 1       -> 1.0 * 2**1 = 2.0")
print("   503 attempt 2       -> 1.0 * 2**2 = 4.0")

Notice `sleep=waits.append`. We **inject** the waiting function so tests run instantly and can *assert* on the waits. Injecting a dependency (a clock, a sleeper, a session) is the single most useful habit for testable code, and you'll write exactly this in exercises 7 and 8.

**What to retry:** timeouts, connection errors, 429, 5xx, and **only idempotent requests** (`GET`). Retrying a `POST` that submits a transaction could do it twice.

**Real sessions can also retry at the connection layer** with `urllib3`'s `Retry` (shown for reference; our helper does the same in plain Python so you can see and test the logic):

```python
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
s = requests.Session()
s.mount("https://", HTTPAdapter(max_retries=Retry(total=4, backoff_factor=1,
                                                  status_forcelist=[429, 500, 502, 503, 504])))
```

## 9. Rate limits

Every free API caps your usage. Know the numbers **before** you design a pipeline:

| Provider | Free-tier limit (per the sources below) | Design implication |
|---|---|---|
| **Etherscan V2** | 5 calls/second, 100,000/day, **shared across all chains on one key** | Throttle to ~3/s; parallel calls to many chains share one budget |
| **CoinGecko Demo** | 10,000 credits/month; a third-party integration doc reports ~100 calls/minute | A call every 10 min per coin ≈ 4,300/month; budget carefully |
| **CoinGecko keyless** | A third-party doc reports roughly 5-15 calls/minute and unpredictable limiting | Fine for a quick test; unreliable for a pipeline |
| **DefiLlama free** | No published hard cap we could verify; be polite | Cache aggressively |

> The Etherscan numbers come from Etherscan's own [rate-limits page](https://docs.etherscan.io/etherscan-v2/rate-limits). The CoinGecko per-minute figures come from a third-party integration doc, not CoinGecko's own page, so **check your plan in the CoinGecko developer dashboard** and treat these as estimates.

### 9.1 Budget before you build

In [ ]:
def monthly_calls(calls_per_run, runs_per_day, days=30):
    return calls_per_run * runs_per_day * days

# CoinGecko Demo = 10,000 credits/month. Refresh 3 coins every 10 minutes?
cost = monthly_calls(calls_per_run=3, runs_per_day=24 * 6)
print(f"3 coins every 10 min  -> {cost:,} calls/month  ({'OVER' if cost > 10_000 else 'within'} 10,000)")

# Refresh 3 coins once per hour instead
cost = monthly_calls(3, 24)
print(f"3 coins every hour    -> {cost:,} calls/month  ({'OVER' if cost > 10_000 else 'within'} 10,000)")

### 9.2 A client-side limiter

Retrying after a 429 is the *reactive* defence; a **limiter** is the *proactive* one: space your calls so you never trip the limit. Our `RateLimiter` sleeps just enough before each call. Here it is with a **fake clock** so you can see the arithmetic without waiting:

In [ ]:
fake_now = [0.0]
slept = []
def fake_sleep(s):
    slept.append(round(s, 3)); fake_now[0] += s          # sleeping advances the fake clock

lim = RateLimiter(calls_per_second=2, clock=lambda: fake_now[0], sleep=fake_sleep)
for i in range(4):
    lim.wait()
    print(f"call {i+1} at t={fake_now[0]:.1f}s")
print("sleeps:", slept)        # first call free; then 0.5s gaps for a 2 calls/sec limit

## 10. Pagination

APIs won't hand you a million rows in one response. They give you a **page** and a way to ask for the next one. This is the API version of `LIMIT`/`OFFSET`.

| Pattern | How it works | SQL parallel | Seen in |
|---|---|---|---|
| **Page number** | `?page=3&per_page=100` | `LIMIT 100 OFFSET 200` | CoinGecko `/coins/markets` |
| **Page + offset (capped)** | `?page=3&offset=100`, but total window limited | `LIMIT/OFFSET` with a cap | Etherscan `txlist` (**page × offset ≤ 10,000**) |
| **Cursor / keyset** | "give me rows *after* this block/id" | `WHERE block > :last ORDER BY block LIMIT n` | Etherscan via `startblock`; many REST APIs |

Cursor pagination is better for big or changing datasets: it doesn't skip or repeat rows when new data arrives mid-crawl (offset paging does), and it has no depth cap. It's also what a database does internally, which is why SQL people call it *keyset pagination*.

### 10.1 First, a Python concept: generators

Pagination is the textbook use of a **generator**. A normal function computes everything and `return`s once. A **generator function** uses `yield` to hand back **one value at a time**, pausing between them and resuming where it left off.

```python
def count_up(n):
    i = 1
    while i <= n:
        yield i        # hand back i, then FREEZE here until asked for the next one
        i += 1
```

In [ ]:
def count_up(n):
    i = 1
    while i <= n:
        print(f"   (generator computing {i})")
        yield i
        i += 1

gen = count_up(3)
print("created:", gen)            # nothing has run yet! A generator is lazy
print("next ->", next(gen))       # runs until the first yield
print("next ->", next(gen))       # resumes, runs to the next yield
print("rest  ->", list(gen))      # exhausts the remainder

Why that matters for APIs: **laziness**. If a wallet has 50,000 transactions, a function that fetches them all before returning makes you wait for every page and hold everything in memory. A generator fetches page 1, **yields its rows immediately**, and only fetches page 2 *if you keep asking*:

In [ ]:
s = FixtureSession()
coins = cg.__class__(s, limiter=limiter(1000)).iter_markets(per_page=5)     # a generator: no HTTP call yet
print("calls so far:", len(s.calls))

first_three = list(itertools.islice(coins, 3))        # islice takes just 3 items
print([c["id"] for c in first_three])
print("calls made  :", len(s.calls), " <- only ONE page was fetched, not all of them")

`itertools.islice(gen, n)` takes the first `n` items without consuming the rest. **Rule of thumb:** if a function produces "a stream of things from somewhere slow", make it a generator. The consumer decides how much to take.

### 10.2 Page-number pagination (CoinGecko)

`iter_markets` loops `page = 1, 2, 3...` and **stops when a page returns fewer rows than asked for** (the last page). That avoids one pointless extra call.

In [ ]:
s = FixtureSession()
client = CoinGeckoClient(s, limiter=limiter(1000))
all_coins = list(client.iter_markets(per_page=5))
print(len(all_coins), "coins in", len(s.calls), "calls (5 + 5 + 2 rows)")
pd.DataFrame(all_coins)[["market_cap_rank", "id", "current_price", "market_cap"]].head(5)

Real CoinGecko caps `per_page` at 250, so use **250** in production (12 coins is 1 call; 12,000 coins is 48), since credits are per *call*, not per row.

### 10.3 Etherscan: pages, a hard cap, and the cursor fix

Page-number paging works for small histories. But Etherscan limits the window: **`page × offset` must stay ≤ 10,000**. Ask for more and it refuses (in the body, with HTTP 200):

In [ ]:
r = session.get("https://api.etherscan.io/v2/api",
                params={"chainid": 1, "module": "account", "action": "txlist", "address": TX_ADDRESS,
                        "page": 101, "offset": 100, "apikey": "demo"}).json()
print(r["status"], "|", r["message"])

Wallets with more than 10,000 transactions are common (any bot, exchange or active DeFi user). The fix is **cursor pagination by block number**: fetch page 1 starting at block `B`, then move `B` forward to the last block you saw:

```
cursor = 0
loop:  rows = txlist(startblock=cursor, page=1, offset=100)
       yield rows
       cursor = last row's blockNumber        # NOT +1: several txs can share a block
```

Because several transactions can live in the **same block**, the boundary block repeats at the start of the next batch, so we **de-duplicate by `hash`**. Both approaches on our 250-transaction demo wallet:

In [ ]:
by_page   = list(es.iter_txlist(TX_ADDRESS, page_size=100))
by_cursor = list(es.iter_txlist_all(TX_ADDRESS, page_size=100))
print("page-number :", len(by_page), "txs")
print("block-cursor:", len(by_cursor), "txs  | unique hashes:", len({t['hash'] for t in by_cursor}))
print("same set    :", {t['hash'] for t in by_page} == {t['hash'] for t in by_cursor})

## 11. Caching: ask once, reuse many times

Three reasons to cache raw responses:
1. **Quota**: a notebook you re-run 30 times shouldn't spend 30× the credits
2. **Speed**: disk is thousands of times faster than the network
3. **Reproducibility**: "the data as I saw it on Tuesday" lets you reproduce a result later

The simplest cache is a file per request with a **TTL** (time to live): reuse it if it's fresher than N seconds. This is the same idea as the `defillama_cache.json` from Week 7, made reusable:

In [ ]:
import hashlib

def cached_json(key, fetch, ttl_s=3600, cache_dir=".cache", now=time.time):
    '''Return cached JSON for `key` if fresher than ttl_s, else call fetch() and store it.'''
    d = pathlib.Path(cache_dir); d.mkdir(exist_ok=True)
    f = d / (hashlib.sha1(key.encode()).hexdigest()[:16] + ".json")
    if f.exists() and now() - f.stat().st_mtime < ttl_s:
        return json.loads(f.read_text())
    data = fetch()
    f.write_text(json.dumps(data))
    return data

with tempfile.TemporaryDirectory() as tmp:             # a throwaway folder for the demo
    s = FixtureSession()
    fetch = lambda: request_json(s, "https://api.llama.fi/protocols")
    a = cached_json("llama:protocols", fetch, cache_dir=tmp)
    b = cached_json("llama:protocols", fetch, cache_dir=tmp)
    print("API calls made:", len(s.calls), "(second read came from disk)")
    print("identical     :", a == b)

Choose TTLs by how fast the data changes: **historical** data (a closed day's price, a mined transaction) **never changes**, so use a huge TTL. **Current** prices want seconds to minutes. Add `.cache/` to `.gitignore`.

## 12. Putting it together: price every transaction in USD

The core move of on-chain analytics is **enriching on-chain facts with off-chain prices**. Etherscan tells you *how much ETH* moved; CoinGecko tells you what ETH was *worth that day*. Join them on the date.

Three data-quality details show up immediately, all from this notebook:
- Etherscan's `value` is a **string of wei**, and often above int64, so we convert with Python `int`
- Etherscan timestamps are **seconds** (strings); CoinGecko's are **milliseconds**
- We join on a **UTC date string** so the two sources line up

In [ ]:
txs = tx_dataframe(list(es.iter_txlist_all(TX_ADDRESS)))
txs["date"] = pd.to_datetime(txs["ts"], unit="s", utc=True).dt.strftime("%Y-%m-%d")
txs["eth"] = txs["value_wei"].map(lambda w: int(w) / 10**18)             # exact int math first, float last

eth_px = cg.market_chart("ethereum", days=60)[["date", "price_usd"]]
enriched = txs.merge(eth_px, on="date", how="left", validate="many_to_one")
enriched["usd_value"] = enriched["eth"] * enriched["price_usd"]

print("transactions:", len(enriched), "| unpriced (no matching date):", enriched["price_usd"].isna().sum())
top = enriched[enriched["is_error"] == 0].nlargest(5, "usd_value")
top.assign(tx=top["hash"].str[:10], to=top["to_addr"].map(shorten_address))[["tx", "date", "eth", "price_usd", "usd_value", "to"]]

In [ ]:
# Daily activity summary: SQL GROUP BY date, in Pandas
daily = (enriched.groupby("date")
                 .agg(txs=("hash", "count"), eth_moved=("eth", "sum"), usd_moved=("usd_value", "sum"),
                      failed=("is_error", "sum"))
                 .assign(usd_moved=lambda d: d["usd_moved"].round(0)))
print(f"total moved: {fmt_large(enriched['usd_value'].sum())} across {len(enriched)} txs "
      f"({enriched['is_error'].mean():.1%} failed)")
daily.head()

In [ ]:
# Gas oracle: Etherscan's live view of current gas (gwei). Compare to Week 8's gas analysis.
print(es.gas_oracle())

## 13. Going live

Everything above ran on fixtures. To hit the real APIs:

1. **Get keys** (free): Etherscan at `etherscan.io/apis` (one key covers all chains via V2); CoinGecko Demo plan from the developer dashboard (recommended even though keyless exists).
2. **Put them in `.env`** (section 6.3) as `ETHERSCAN_API_KEY` and `COINGECKO_DEMO_API_KEY`, and make sure `.env` is in `.gitignore`.
3. **Run the smoke test** from this folder. It checks each endpoint, key and response shape and prints a clear PASS/FAIL per source; **run it before debugging anything else**:
   ```bash
   python3 live_smoke_test.py
   ```
4. **Flip the flag** in the Setup cell: `LIVE = True`. The same code now uses real HTTP, real limits and your keys.

Expect small differences from the fixtures: extra fields, different numbers, occasional empty pages. If a field name differs, the **official docs win** over this course. Update the client and note it in your README. (That is real data engineering.)

> **Honest note:** the live smoke test and `LIVE=True` path were written from the providers' documentation but **could not be executed in the sandbox this course was built in**, which has no access to those APIs. If something fails, the smoke test tells you which source and why.

## 14. Common pitfalls

| Pitfall | Symptom | Fix |
|---|---|---|
| No `timeout=` | Script hangs forever | Always pass a timeout |
| Trusting HTTP 200 (Etherscan) | "Successful" call, empty/garbage data | Check `status` in the body |
| Retrying every error | Hammering a server that said 401 | Retry only 429/5xx/network |
| Ignoring `Retry-After` | Repeated 429s, temporary bans | Obey it when present |
| Timestamps in the wrong unit | Dates in 1970 or 58,000 AD | Know each provider's unit |
| Numbers as strings | `"123" + "4"` = `"1234"` | Convert explicitly, big ints with `int()` |
| Key in code / query-string logs | Leaked key | Env vars, header auth, `redact()` |
| Fetching all pages into a list first | Slow and memory-hungry | Generators + `islice` |
| `page` pagination past a cap | Silent truncation or an error | Cursor by block |
| No caching during development | Burned quota | `cached_json` |
| Float math on wei | Lost digits | Python `int`, convert last |

## Lesson summary

| Concept | SQL / Dune idea | Python |
|---|---|---|
| Endpoint + params | table + `WHERE` | `requests` + `params={}` |
| Response | result rows | `resp.json()` → dict/list |
| JSON → table | result set | `pd.DataFrame`, `json_normalize` |
| Status codes | query errors | check `status_code`; 429/5xx retry, else fail |
| API key | account | env var + header; never in code |
| Body-level errors | n/a | Etherscan `status: "0"` |
| Retries | re-run query | exponential backoff + `Retry-After` |
| Rate limits | query credits | `RateLimiter`, budgets |
| Pagination | `LIMIT/OFFSET`, keyset | page loop; block cursor |
| Generators | cursor/streaming | `yield`, `islice` |
| Caching | materialised results | file cache with TTL |
| Enrichment | `JOIN prices` | `merge` on a UTC date |

**Key ideas:** one client per provider, one exception type, retry only what's transient, never trust a 200, never hardcode a key, never float a wei.

> 📺 **Watch / 📖 Read next** (full curated list with study order in `RESOURCES.md`): Corey Schafer's video *"Python Requests Tutorial: Request Web Pages, Download Images, POST Data, Read JSON, and More"* (search the exact title on YouTube); the free book *Python for Everybody*, Chapter 13 *Using Web Services* (`pythonlearn.com`); and the official [CoinGecko](https://docs.coingecko.com/demo/reference/authentication) and [Etherscan V2](https://docs.etherscan.io/etherscan-v2/rate-limits) docs.

## What's next

**`lesson_expanded.ipynb`**: you now have data in memory. Next you'll persist it in **SQLite**: schemas with constraints, safe parameterised queries, upserts, window functions, and the Pandas ↔ SQL bridge.

## Commit your work

```bash
cd ~/Desktop/python-for-blockchain-analytics
git add phase-2-python-for-data/week-10-apis-sql/
git commit -m "phase-2/week-10: APIs and SQL from Python"
git push
```

Before committing, confirm no key or `.env` is staged: `git status` should show neither.